<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #0284c7; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Agrupaciones, Joins y Funciones de Ventana 📊🔗
      </h1>
      <p style="margin: 6px 0 0 0; color: #0284c7; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Polars de Alto Rendimiento
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #0284c7; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 11 Extra
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #0284c7; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/11%20-%20Polars/02_Agrupaciones_Joins_y_Funciones_Ventana.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. Agrupaciones con `group_by()` y Agregación Multihilo `.agg()` 👥

En Polars moderno, la sintaxis estándar es `group_by(...)` (en lugar de `groupby`, heredado de Pandas). Dentro del contexto `.agg()`, podemos calcular decenas de métricas simultáneas, y cada una se reparte entre los núcleos disponibles del procesador.

In [37]:
import polars as pl
import numpy as np
import os

import os, urllib.request, urllib.parse

def load_dataset(filename, module_name="11 - Polars"):
    """
    Carga o descarga de forma segura el dataset para ejecución local o en Google Colab.
    Si no se encuentra localmente ni en GitHub, lo genera automáticamente.
    """
    candidates = [
        os.path.join("data", filename),
        os.path.join(module_name, "data", filename),
        os.path.join("..", "data", filename),
        os.path.join("..", module_name, "data", filename),
        os.path.join("Data Science programming", module_name, "data", filename),
        os.path.join("..", "Data Science programming", module_name, "data", filename),
        filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
            
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    folder_path = f"Data Science programming/{module_name}"
    encoded_folder = urllib.parse.quote(folder_path)
    encoded_file = urllib.parse.quote(filename)
    
    urls = [
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{encoded_folder}/data/{encoded_file}",
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/master/{encoded_folder}/data/{encoded_file}"
    ]
    
    print(f"📥 Intentando descargar '{filename}' desde el repositorio oficial...")
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    with open(target_path, 'wb') as out_f:
                        out_f.write(response.read())
                    print(f"✅ Dataset '{filename}' descargado exitosamente.")
                    return target_path
        except Exception:
            continue
            
    print(f"⚙️ Generando '{filename}' sintéticamente para ejecución inmediata...")
    import polars as pl
    import numpy as np
    np.random.seed(42)
    
    n_clientes = 1000
    df_c = pl.DataFrame({
        'id_cliente': [f'CLI-{i:04d}' for i in range(1, n_clientes + 1)],
        'nombre': [f'Cliente_{i}' for i in range(1, n_clientes + 1)],
        'segmento': np.random.choice(['Corporativo', 'Pyme', 'Consumo', 'Gobierno'], n_clientes),
        'edad': np.random.randint(18, 70, n_clientes),
        'ciudad_residencia': np.random.choice(['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Bucaramanga'], n_clientes),
        'ingreso_anual': np.random.normal(45000000, 15000000, n_clientes).round(2)
    })
    
    n_ventas = 60000
    cats = ['Tecnología', 'Mobiliario', 'Material de Oficina', 'Servicios']
    prods = ['Laptop Pro', 'Monitor 4K', 'Silla Ergonómica', 'Escritorio', 'Papel A4', 'Tóner', 'Mantenimiento']
    ciudades = ['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Barranquilla']
    
    cant = np.random.randint(1, 10, n_ventas)
    pu = np.random.choice([25000.0, 120000.0, 450000.0, 1200000.0, 3500000.0], n_ventas)
    desc = np.random.choice([0.0, 0.05, 0.10, 0.15], n_ventas)
    tot = (cant * pu * (1 - desc)).round(2)
    
    df_v = pl.DataFrame({
        'id_venta': [f'VNT-{i:06d}' for i in range(1, n_ventas + 1)],
        'fecha': [f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}' for _ in range(n_ventas)],
        'id_cliente': np.random.choice(df_c['id_cliente'], n_ventas),
        'categoria': np.random.choice(cats, n_ventas),
        'producto': np.random.choice(prods, n_ventas),
        'cantidad': cant,
        'precio_unitario': pu,
        'descuento': desc,
        'ciudad_venta': np.random.choice(ciudades, n_ventas),
        'total_venta': tot
    })
    
    c_csv_path = os.path.join("data", "clientes.csv")
    c_pq_path = os.path.join("data", "clientes.parquet")
    v_csv_path = os.path.join("data", "ventas.csv")
    v_pq_path = os.path.join("data", "ventas.parquet")
    
    if not os.path.exists(c_csv_path): df_c.write_csv(c_csv_path)
    if not os.path.exists(c_pq_path): df_c.write_parquet(c_pq_path)
    if not os.path.exists(v_csv_path): df_v.write_csv(v_csv_path)
    if not os.path.exists(v_pq_path): df_v.write_parquet(v_pq_path)
    
    print(f"✅ Datasets preparados exitosamente en '{target_path}'.")
    return target_path

df_ventas = pl.read_csv(load_dataset("ventas.csv"))
df_clientes = pl.read_csv(load_dataset("clientes.csv"))
print(f"🚀 Polars versión: {pl.__version__}")
print(f"Ventas: {df_ventas.shape} | Clientes: {df_clientes.shape}")

🚀 Polars versión: 1.35.2
Ventas: (60000, 10) | Clientes: (1000, 6)


> 💡 `pl.len()` cuenta filas por grupo (equivalente al `COUNT(*)` de SQL). Nota también que dentro de `.agg()` puedes anidar un `.filter()` — aquí `pl.col("total_venta").filter(pl.col("descuento") > 0).count()` cuenta, *dentro de cada grupo*, solo las filas con descuento aplicado, sin necesidad de un `group_by` separado.

---
## 2. Agrupando por Varias Columnas a la Vez 🧮

`group_by()` acepta una lista de columnas: cada combinación única de valores forma un grupo distinto. Es útil para responder preguntas más finas, como "¿cuánto vendió cada categoría, en cada ciudad?".

In [38]:
resumen_cat_ciudad = df_ventas.group_by(["categoria", "ciudad_venta"]).agg([
    pl.len().alias("num_ventas"),
    pl.col("total_venta").sum().alias("ingreso_total")
]).sort(["categoria", "ingreso_total"], descending=[False, True])

display(resumen_cat_ciudad.head(8))

categoria,ciudad_venta,num_ventas,ingreso_total
str,str,u32,f64
"""Material de Oficina""","""Medellín""",3015,1.5587e10
"""Material de Oficina""","""Bogotá""",3042,1.5516e10
"""Material de Oficina""","""Cali""",3020,1.4708e10
"""Material de Oficina""","""Barranquilla""",3042,1.4695e10
"""Material de Oficina""","""Tunja""",2960,1.4104e10
"""Mobiliario""","""Bogotá""",3067,1.5588e10
"""Mobiliario""","""Medellín""",3027,1.4793e10
"""Mobiliario""","""Tunja""",2956,1.4736e10


### El Patrón "Top-N por Grupo"

Una necesidad muy común es encontrar, para cada grupo, solo su fila más relevante (por ejemplo, el producto que más ingresos generó *dentro de cada categoría*). Se resuelve combinando `group_by().agg()`, un `sort()` y un segundo `group_by(..., maintain_order=True).head(n)`:

In [39]:
top_producto_por_categoria = (
    df_ventas
    .group_by(["categoria", "producto"])
    .agg(pl.col("total_venta").sum().alias("ingreso"))
    .sort("ingreso", descending=True)
    .group_by("categoria", maintain_order=True)
    .head(1)
)
display(top_producto_por_categoria)

categoria,producto,ingreso
str,str,f64
"""Material de Oficina""","""Papel A4""",1.1840e10
"""Mobiliario""","""Escritorio""",1.1574e10
"""Tecnología""","""Papel A4""",1.1209e10
"""Servicios""","""Mantenimiento""",1.0787e10


---
## 3. Combinación y Fusión de Datos: Tipos de Joins 🔗

Polars ofrece algoritmos de hash join en Rust extremadamente rápidos. Los 4 tipos que más vas a usar:

| `how=` | ¿Qué hace? | Analogía |
|---|---|---|
| `"inner"` | Conserva solo las filas cuya llave existe en **ambas** tablas. | La intersección de dos listas. |
| `"left"` | Conserva **todas** las filas de la tabla izquierda; rellena con `null` donde no hay match. | "Completa mi lista con datos de la otra, si existen". |
| `"semi"` | Filtra filas de la izquierda que **sí** tienen match en la derecha, pero sin traer columnas de la derecha. | "¿Quién de mi lista también está en la otra?" (sin fusionar datos). |
| `"anti"` | Filtra filas de la izquierda que **no** tienen match en la derecha. | "¿Quién de mi lista NO está en la otra?" |

En todos los casos usaremos `ventas` (izquierda) y `clientes` (derecha), unidas por la llave compartida `id_cliente`.

### 3.1. `left` join: Enriquecer Ventas con Datos del Cliente

In [40]:
df_completo = df_ventas.join(
    df_clientes,
    on="id_cliente",
    how="left"
)
print(f"Dimensiones tras el join: {df_completo.shape}")
display(df_completo.select(["id_venta", "id_cliente", "nombre", "segmento", "ciudad_residencia", "total_venta"]).head(4))

Dimensiones tras el join: (60000, 15)


id_venta,id_cliente,nombre,segmento,ciudad_residencia,total_venta
str,str,str,str,str,f64
"""VNT-000001""","""CLI-0117""","""Cliente_117""","""Corporativo""","""Tunja""",166250.0
"""VNT-000002""","""CLI-0775""","""Cliente_775""","""Pyme""","""Tunja""",1.08e6
"""VNT-000003""","""CLI-0705""","""Cliente_705""","""Corporativo""","""Cali""",106250.0
"""VNT-000004""","""CLI-0592""","""Cliente_592""","""Gobierno""","""Bucaramanga""",1.71e6


### 3.2. `inner` join: Solo Ventas de Clientes Corporativos

Si en lugar de unir contra la tabla completa de `clientes` la filtramos primero, un `inner` join actúa como un filtro compuesto: solo sobreviven las ventas cuyo `id_cliente` aparece en el subconjunto de la derecha.

In [41]:
clientes_corporativo = df_clientes.filter(pl.col("segmento") == "Corporativo")

ventas_corporativas = df_ventas.join(clientes_corporativo, on="id_cliente", how="inner")
print(f"Ventas de clientes Corporativo: {ventas_corporativas.height} de {df_ventas.height} totales")
display(ventas_corporativas.select(["id_venta", "nombre", "segmento", "total_venta"]).head(4))

Ventas de clientes Corporativo: 15451 de 60000 totales


id_venta,nombre,segmento,total_venta
str,str,str,f64
"""VNT-000001""","""Cliente_117""","""Corporativo""",166250.0
"""VNT-000003""","""Cliente_705""","""Corporativo""",106250.0
"""VNT-000006""","""Cliente_644""","""Corporativo""",3.4425e6
"""VNT-000009""","""Cliente_776""","""Corporativo""",540000.0


### 3.3. `semi` y `anti` join: Filtrar sin Fusionar Columnas

`semi` y `anti` son, en el fondo, **filtros** que usan otra tabla como criterio — nunca agregan columnas de la derecha, por lo que el número de columnas del resultado es igual al de la tabla izquierda.

In [42]:
# SEMI: ventas hechas por clientes Corporativo (mismo resultado que el inner de arriba,
# pero sin traer columnas de clientes — solo las columnas originales de ventas)
ventas_semi_corp = df_ventas.join(clientes_corporativo, on="id_cliente", how="semi")
print(f"Semi join -> filas: {ventas_semi_corp.height} | columnas: {ventas_semi_corp.columns}")

# ANTI: ventas hechas por clientes que NO pertenecen al segmento Gobierno
clientes_gobierno = df_clientes.filter(pl.col("segmento") == "Gobierno")
ventas_no_gobierno = df_ventas.join(clientes_gobierno, on="id_cliente", how="anti")
print(f"\nAnti join -> ventas de clientes no-Gobierno: {ventas_no_gobierno.height} de {df_ventas.height} totales")

# ANTI también sirve como chequeo de integridad: ¿hay clientes registrados sin NINGUNA compra?
clientes_sin_compras = df_clientes.join(df_ventas, on="id_cliente", how="anti")
print(f"Clientes sin ninguna compra registrada: {clientes_sin_compras.height} de {df_clientes.height} totales")

Semi join -> filas: 15451 | columnas: ['id_venta', 'fecha', 'id_cliente', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento', 'ciudad_venta', 'total_venta']

Anti join -> ventas de clientes no-Gobierno: 43044 de 60000 totales
Clientes sin ninguna compra registrada: 0 de 1000 totales


> 💡 Que `clientes_sin_compras` dé **0** filas es en sí mismo un resultado útil: confirma que, en este dataset, todos los clientes registrados tienen al menos una compra — exactamente el tipo de validación de integridad para el que se usa `anti` join en un pipeline real.

---
## 4. Funciones de Ventana (*Window Functions*) con `.over()` 🪟⚡

Las funciones de ventana permiten calcular estadísticas agregadas sobre particiones **sin colapsar filas** ni hacer un `group_by` destructivo: el resultado agregado se "pega" de vuelta en cada fila original, dentro de su grupo.

In [43]:
# Promedio de la categoría y participación de cada venta dentro de su categoría, SIN group_by
df_ventana = df_ventas.select([
    pl.col("id_venta"),
    pl.col("categoria"),
    pl.col("total_venta"),
    pl.col("total_venta").mean().over("categoria").alias("media_categoria"),
    (pl.col("total_venta") / pl.col("total_venta").sum().over("categoria") * 100).round(4).alias("pct_ingreso_categoria")
])
display(df_ventana.head(6))

id_venta,categoria,total_venta,media_categoria,pct_ingreso_categoria
str,str,f64,f64,f64
"""VNT-000001""","""Mobiliario""",166250.0,4.9506e6,0.0002
"""VNT-000002""","""Mobiliario""",1.08e6,4.9506e6,0.0015
"""VNT-000003""","""Mobiliario""",106250.0,4.9506e6,0.0001
"""VNT-000004""","""Mobiliario""",1.71e6,4.9506e6,0.0023
"""VNT-000005""","""Servicios""",200000.0,4.9565e6,0.0003
"""VNT-000006""","""Material de Oficina""",3.4425e6,4.9480e6,0.0046


### `group_by()` vs. `.over()`: la diferencia clave

| | `group_by().agg()` | `.select(...).over(...)` |
|---|---|---|
| **Filas de salida** | Una por grupo (colapsa) | Una por fila original (no colapsa) |
| **Uso típico** | Un resumen/reporte agregado | "Pegar" un valor de grupo a cada fila individual, para comparar |

### Más patrones con `.over()`: Ranking y Acumulados

`.over()` no se limita a promedios: cualquier expresión (`.rank()`, `.cum_sum()`, `.max()`, ...) puede evaluarse "dentro de" una partición.

In [44]:
# Top 3 ventas (por monto) dentro de cada categoría, usando .rank().over()
top3_por_categoria = (
    df_ventas
    .select([
        pl.col("id_venta"),
        pl.col("categoria"),
        pl.col("total_venta"),
        pl.col("total_venta").rank(method="ordinal", descending=True).over("categoria").alias("ranking_en_categoria")
    ])
    .filter(pl.col("ranking_en_categoria") <= 3)
    .sort(["categoria", "ranking_en_categoria"])
)
display(top3_por_categoria)

# Gasto acumulado por cliente, ordenado en el tiempo (útil para curvas de gasto acumulado)
gasto_acumulado = (
    df_ventas
    .sort(["id_cliente", "fecha"])
    .select([
        pl.col("id_cliente"),
        pl.col("fecha"),
        pl.col("total_venta"),
        pl.col("total_venta").cum_sum().over("id_cliente").alias("gasto_acumulado_cliente")
    ])
)
display(gasto_acumulado.head(6))

id_venta,categoria,total_venta,ranking_en_categoria
str,str,f64,u32
"""VNT-000210""","""Material de Oficina""",3.15e7,1
"""VNT-002782""","""Material de Oficina""",3.15e7,2
"""VNT-002988""","""Material de Oficina""",3.15e7,3
"""VNT-000412""","""Mobiliario""",3.15e7,1
"""VNT-001106""","""Mobiliario""",3.15e7,2
…,…,…,…
"""VNT-000168""","""Servicios""",3.15e7,2
"""VNT-001310""","""Servicios""",3.15e7,3
"""VNT-000611""","""Tecnología""",3.15e7,1


id_cliente,fecha,total_venta,gasto_acumulado_cliente
str,str,f64,f64
"""CLI-0001""","""2024-01-03""",2.295e6,2.295e6
"""CLI-0001""","""2024-01-10""",1.53e6,3.825e6
"""CLI-0001""","""2024-01-11""",2.8e7,3.1825e7
"""CLI-0001""","""2024-01-15""",2.835e7,6.0175e7
"""CLI-0001""","""2024-01-15""",3.6e6,6.3775e7
"""CLI-0001""","""2024-02-05""",2.7e6,6.6475e7


---
## 5. Práctica Integrada: Agrupar, Unir y Comparar en una Sola Pipeline 🧩

Combinemos los tres temas del cuaderno: unimos `ventas` con `clientes` (`join`), resumimos por segmento (`group_by`) y, en paralelo, comparamos cada venta contra el promedio de su ciudad de residencia (`.over()`).

In [45]:
pipeline = (
    df_ventas
    .join(df_clientes, on="id_cliente", how="left")
    .with_columns(
        pl.col("total_venta").mean().over("ciudad_residencia").round(2).alias("promedio_ciudad_residencia")
    )
    .with_columns(
        (pl.col("total_venta") - pl.col("promedio_ciudad_residencia")).round(2).alias("diferencia_vs_promedio")
    )
)

resumen_por_segmento = pipeline.group_by("segmento").agg([
    pl.len().alias("num_ventas"),
    pl.col("total_venta").sum().alias("ingreso_total"),
    pl.col("diferencia_vs_promedio").mean().round(2).alias("desviacion_promedio_vs_ciudad")
]).sort("ingreso_total", descending=True)

display(resumen_por_segmento)

segmento,num_ventas,ingreso_total,desviacion_promedio_vs_ciudad
str,u32,f64,f64
"""Gobierno""",16956,8.3932e10,2422.89
"""Corporativo""",15451,7.6634e10,15574.72
"""Pyme""",13724,6.8326e10,30527.59
"""Consumo""",13869,6.7894e10,-50521.88


---
## 6. Ejercicio Práctico: Ranking de Clientes por Ciudad 🧪

Usando `df_ventas` y `df_clientes`, resuelve lo siguiente:

1. Une (`join`, `how="left"`) `df_ventas` con `df_clientes` por `id_cliente`.
2. Con `group_by(["ciudad_residencia", "id_cliente", "nombre"])` y `.agg()`, calcula el **gasto total** (`total_venta.sum()`) por cliente.
3. Usando `.over("ciudad_residencia")` sobre ese resultado agregado, agrega una columna `ranking_en_ciudad` con `.rank(method="ordinal", descending=True)` que indique la posición de cada cliente dentro de su ciudad según su gasto total.
4. Filtra y muestra solo los clientes con `ranking_en_ciudad <= 3` (el podio de cada ciudad), ordenado por `ciudad_residencia` y `ranking_en_ciudad`.

Escribe tu solución en la celda de abajo antes de revisar la respuesta guiada:

In [46]:
# 1-2. Join + group_by + agg: gasto total por cliente
# gasto_por_cliente = ...

# 3. Ranking dentro de cada ciudad con .over()
# gasto_con_ranking = ...

# 4. Filtrar el podio (top 3) de cada ciudad
# podio_por_ciudad = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
gasto_por_cliente = (
    df_ventas
    .join(df_clientes, on="id_cliente", how="left")
    .group_by(["ciudad_residencia", "id_cliente", "nombre"])
    .agg(pl.col("total_venta").sum().alias("gasto_total"))
)

gasto_con_ranking = gasto_por_cliente.with_columns(
    pl.col("gasto_total").rank(method="ordinal", descending=True).over("ciudad_residencia").alias("ranking_en_ciudad")
)

podio_por_ciudad = (
    gasto_con_ranking
    .filter(pl.col("ranking_en_ciudad") <= 3)
    .sort(["ciudad_residencia", "ranking_en_ciudad"])
)
display(podio_por_ciudad)
```
</details>

---
## 7. Resumen y Próximos Pasos 📌

| Concepto | Idea Clave |
|---|---|
| **`group_by().agg()`** | Resume filas en grupos; una fila de salida por grupo. Multihilo por defecto. |
| **`pl.len()`** | Cuenta filas por grupo (equivalente a `COUNT(*)`). |
| **Agregación condicional** | `pl.col(x).filter(cond).agg_fn()` calcula una métrica solo sobre un subconjunto, dentro de `.agg()`. |
| **Patrón Top-N por grupo** | `sort()` + `group_by(..., maintain_order=True).head(n)`. |
| **`join(how="inner")`** | Solo filas con llave en ambas tablas. |
| **`join(how="left")`** | Todas las filas de la izquierda; `null` donde no hay match. |
| **`join(how="semi")`** | Filtra la izquierda por existencia en la derecha, sin traer sus columnas. |
| **`join(how="anti")`** | Filtra la izquierda por **ausencia** en la derecha (también sirve como chequeo de integridad). |
| **`.over(columna)`** | Calcula una agregación por partición **sin** colapsar filas — el valor se "pega" en cada fila del grupo. |
| **`group_by` vs. `.over()`** | El primero colapsa filas (resumen); el segundo las conserva (comparación fila a fila contra su grupo). |

➡️ Con esto completas el módulo extra de Polars: dominas los 3 contextos de expresiones, las agregaciones multihilo, los 4 tipos de join más usados y las funciones de ventana — las piezas centrales para transformar datos tabulares grandes de forma rápida y declarativa.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Módulo Extra: Polars de Alto Rendimiento</i>
  </p>
</div>